# Criação da Base de Dados

In [1]:
# @title ## Set Up the Environment
!pip install dash plotly pandas sqlite-utils

##

In [2]:
# @title ## 1. Objetivos

Criar o banco de dados da DataConneti iremos utilizar o banco SQLite

In [3]:
# @title ## 2. Importação das Bibliotecas
import pandas as pd
import sqlite3
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import Markdown

import os
import math
import glob


In [4]:
# @title ## 3. Carregamentos dos Dados
df_clientes = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/raw/dc_clientes.csv')
df_apontamentos = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/raw/dc_apontamentos.csv')
df_projetos = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/raw/dc_projetos.csv')
df_analistas = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/raw/dc_analistas.csv')
df_satisfacao = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/raw/dc_satisfacao.csv')

In [5]:
# @title ## 4. Validação dos Dados
print("=" * 60)
print("VALIDAÇÃO DOS DADOS")
print("=" * 60)
print(f"df_clientes: {df_clientes.shape}")
print(f"df_apontamentos: {df_apontamentos.shape}")
print(f"df_projetos: {df_projetos.shape}")
print(f"df_satisfacao: {df_satisfacao.shape}")
print(f"df_analistas: {df_analistas.shape}")

VALIDAÇÃO DOS DADOS
df_clientes: (62, 7)
df_apontamentos: (9879, 6)
df_projetos: (141, 11)
df_satisfacao: (103, 5)
df_analistas: (24, 6)


In [6]:
# prompts
# """
# criar o banco de dados da DataConnect utilizando sqlite para ser populado
#"""

## 5. Criação e População do Banco de Dados SQLite

Vamos criar um banco de dados SQLite chamado `dataconnect.db` e carregar cada DataFrame em uma tabela correspondente dentro deste banco de dados. Isso facilitará a realização de consultas e análises SQL posteriormente.

In [7]:
DB_NAME = 'dataconnect.db'

# Conectar ao banco de dados SQLite (ele será criado se não existir)
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# Dicionário de dataframes para facilitar a iteração
dataframes = {
    'clientes': df_clientes,
    'apontamentos': df_apontamentos,
    'projetos': df_projetos,
    'analistas': df_analistas,
    'satisfacao': df_satisfacao
}

# Salvar cada dataframe como uma tabela no banco de dados
for table_name, df in dataframes.items():
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Tabela '{table_name}' criada e populada com sucesso.")

# Verificar as tabelas criadas no banco de dados
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("\nTabelas no banco de dados:")
for table in tables:
    print(table[0])

# Fechar a conexão com o banco de dados
conn.close()
print("\nConexão com o banco de dados fechada.")

Tabela 'clientes' criada e populada com sucesso.
Tabela 'apontamentos' criada e populada com sucesso.
Tabela 'projetos' criada e populada com sucesso.
Tabela 'analistas' criada e populada com sucesso.
Tabela 'satisfacao' criada e populada com sucesso.

Tabelas no banco de dados:
clientes
apontamentos
projetos
analistas
satisfacao

Conexão com o banco de dados fechada.


## 6. Salvando o Banco de Dados no Diretório `data/raw`



In [8]:
import os
import shutil # Import shutil for cross-filesystem moves

# Caminho atual do banco de dados (no diretório raiz do Colab)
source_path = os.path.join('/content/', DB_NAME)

# Caminho de destino
destination_dir = '/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/raw'
destination_path = os.path.join(destination_dir, DB_NAME)

# Criar o diretório de destino se ele não existir
if not os.path.exists(destination_dir):
    os.makedirs(destination_dir)
    print(f"Diretório '{destination_dir}' criado.")

# Mover o arquivo do banco de dados usando shutil.move para lidar com diferentes filesystems
if os.path.exists(source_path):
    shutil.move(source_path, destination_path)
    print(f"O banco de dados '{DB_NAME}' foi movido para '{destination_path}'.")
else:
    print(f"Erro: O arquivo '{DB_NAME}' não foi encontrado em '{source_path}'.")

# Verificar a existência do arquivo no novo local
if os.path.exists(destination_path):
    print("Verificação: Arquivo encontrado no diretório de destino.")
else:
    print("Verificação: Arquivo NÃO encontrado no diretório de destino.")

O banco de dados 'dataconnect.db' foi movido para '/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/raw/dataconnect.db'.
Verificação: Arquivo encontrado no diretório de destino.


                    ┌──────────────┐
                    │   CLIENTES   │
                    └──────┬───────┘
                           │
                           │ cliente_id
                           ▼
                    ┌──────────────┐
                    │   PROJETOS   │
                    └───┬──────┬───┘
                        │      │
             projeto_id │      │ projeto_id
                        ▼      ▼
              ┌────────────┐ ┌───────────────┐
              │APONTAMENTOS│ │ SATISFAÇÃO    │
              └─────┬──────┘ └───────────────┘
                    │
                    │ analista_id
                    ▼
              ┌─────────────┐
              │  ANALISTAS  │
              └─────────────┘

In [9]:
# @title ## Informações Iniciais do dataset - dc_clientes
display(Markdown('### **Primeiras Linhas do Dataset**'))
display(df_clientes.head())

display(Markdown('### **Ultimas Linhas do Dataset**'))
display(df_clientes.tail())

display(Markdown('### **Informações do Dataset**'))
display(df_clientes.info())

display(Markdown('### **Quantidade de Linhas e Colunas do Dataset**'))
display(df_clientes.shape)

display(Markdown('### **Quantidade de Valores Ausentes**'))
display(df_clientes.isnull().sum())

display(Markdown('### **Quantidade de Valores Duplicados**'))
display(df_clientes.duplicated().sum())

display(Markdown('### **Quantidade de Valores Únicos**'))
display(df_clientes.nunique())

display(Markdown('### **Estatísticas Descritivas**'))
display(df_clientes.describe())

display(Markdown('### **Os tipos de dados**'))
display(df_clientes.dtypes)

### **Primeiras Linhas do Dataset**

,cliente_id,cliente,setor,porte,cidade,uf,data_cadastro
0,CLI001,Umbu S.A.,NaN,Pequeno,Rio de Janeiro,Rio de Janeiro,2021-10-13
1,CLI002,Xingu S.A.,Indústria,Médio,Florianópolis,SC,2021-03-07
2,CLI003,Aurora S.A.,Saúde,Médio,Curitiba,PR,2021-02-24
3,CLI004,Rubi Ltda,Logística,Pequeno,Florianópolis,SC,2024-04-21
4,CLI005,Ipê S.A.,Logística,Pequeno,Belo Horizonte,MG,15/11/2021


### **Ultimas Linhas do Dataset**

,cliente_id,cliente,setor,porte,cidade,uf,data_cadastro
57,CLI058,Rubi Participações,Serviços Financeiros,Pequeno,São Paulo,SP,2025-12-28
58,CLI059,Dunas Brasil,Serviços Financeiros,Pequeno,Florianópolis,SC,2024-05-23
59,CLI060,Kairós Ltda,Tecnologia,Médio,Porto Alegre,RS,2026-01-29
60,CLI004,Rubi Ltda,Logística,Pequeno,Florianópolis,SC,2024-04-21
61,CLI028,Rubi Holding,SETOR PÚBLICO,Pequeno,Rio de Janeiro,RJ,02/09/2021


### **Informações do Dataset**

...

,0
cliente_id,60
cliente,60
setor,17
porte,3
cidade,10
uf,19
data_cadastro,60


### **Estatísticas Descritivas**

,cliente_id,cliente,setor,porte,cidade,uf,data_cadastro
count,62,62,57,62,62,62,62
unique,60,60,17,3,10,19,60
top,CLI004,Rubi Ltda,Saúde,Pequeno,São Paulo,SP,2024-04-21
freq,2,2,7,38,11,11,2


### **Os tipos de dados**

,0
cliente_id,object
cliente,object
setor,object
porte,object
cidade,object
uf,object
data_cadastro,object


In [10]:
# @title ## Informações Iniciais do dataset - dc_analistas
display(Markdown('### **Primeiras Linhas do Dataset**'))
display(df_analistas.head())

display(Markdown('### **Ultimas Linhas do Dataset**'))
display(df_analistas.tail())

display(Markdown('### **Informações do Dataset**'))
display(df_analistas.info())

display(Markdown('### **Quantidade de Linhas e Colunas do Dataset**'))
display(df_analistas.shape)

display(Markdown('### **Quantidade de Valores Ausentes**'))
display(df_analistas.isnull().sum())

display(Markdown('### **Quantidade de Valores Duplicados**'))
display(df_analistas.duplicated().sum())

display(Markdown('### **Quantidade de Valores Únicos**'))
display(df_analistas.nunique())

display(Markdown('### **Estatísticas Descritivas**'))
display(df_analistas.describe())

display(Markdown('### **Os tipos de dados**'))
display(df_analistas.dtypes)

### **Primeiras Linhas do Dataset**

,analista_id,analista,squad,senioridade,custo_hora,data_admissao
0,ANL001,Ana Barbosa,ALPHA,Senior,"150,00",2021-12-07
1,ANL002,Bruno Ipiranga,Bravo,Pleno,95.00,2021-08-30
2,ANL003,Carla Esteves,Charlie,Junior,55.00,2024-12-26
3,ANL004,Diego Freitas,Delta,Especialista,210.00,2025-07-28
4,ANL005,Elisa Werneck,Echo,Pleno,95.00,24/10/2023


### **Ultimas Linhas do Dataset**

,analista_id,analista,squad,senioridade,custo_hora,data_admissao
19,ANL020,Thiago Freitas,Bravo,Junior,55.00,30-out-2025
20,ANL021,Úrsula Marques,Charlie,Senior,"150,00",08/19/2021
21,ANL022,Vinícius Henriques,Delta,Senior,150.00,2022-07-14
22,ANL023,Wanda Oliveira,Echo,Senior,150.00,2023-05-18
23,ANL024,Yuri Henriques,Foxtrot,Junior,55.00,2022-08-31


### **Informações do Dataset**

...

,0
analista_id,24
analista,24
squad,9
senioridade,4
custo_hora,7
data_admissao,24


### **Estatísticas Descritivas**

,analista_id,analista,squad,senioridade,custo_hora,data_admissao
count,24,24,24,24,24,24
unique,24,24,9,4,7,24
top,ANL001,Ana Barbosa,Bravo,Junior,55.00,2021-12-07
freq,1,1,4,7,7,1


### **Os tipos de dados**

,0
analista_id,object
analista,object
squad,object
senioridade,object
custo_hora,object
data_admissao,object


In [11]:
# @title ## Informações Iniciais do dataset - dc_projetos
display(Markdown('### **Primeiras Linhas do Dataset**'))
display(df_projetos.head())

display(Markdown('### **Ultimas Linhas do Dataset**'))
display(df_projetos.tail())

display(Markdown('### **Informações do Dataset**'))
display(df_projetos.info())

display(Markdown('### **Quantidade de Linhas e Colunas do Dataset**'))
display(df_projetos.shape)

display(Markdown('### **Quantidade de Valores Ausentes**'))
display(df_projetos.isnull().sum())

display(Markdown('### **Quantidade de Valores Duplicados**'))
display(df_projetos.duplicated().sum())

display(Markdown('### **Quantidade de Valores Únicos**'))
display(df_projetos.nunique())

display(Markdown('### **Estatísticas Descritivas**'))
display(df_projetos.describe())

display(Markdown('### **Os tipos de dados**'))
display(df_projetos.dtypes)

### **Primeiras Linhas do Dataset**

,projeto_id,cliente_id,nome_projeto,tipo_servico,squad,status,data_inicio,data_fim_prevista,data_fim_real,horas_vendidas,valor_contrato
0,PRJ0001,CLI013,Dashboard BI - Windsor,Dashboard B.I.,Delta,CONCLUÍDO,2024-03-19,2024-05-06,08/05/2024,288,NaN
1,PRJ0002,CLI022,Diagnóstico de Dados - Umbu,Diagnóstico de Dados,Alpha,Cancelado,2024-10-01,2024-10-28,NaN,89,24044.51
2,PRJ0003,CLI051,Pipeline de Dados - Cristal,Pipeline de Dados,Charlie,Concluído,2025-06-16,20/10/2025,10/15/2025,423,91525.58
3,PRJ0004,CLI060,Dashboard BI - Kairós,Dashboard BI,Echo,Concluído,2024-03-19,25-abr-2024,2024-04-26,220,47409.01
4,PRJ0005,CLI020,Modelo Preditivo - Duna,Modelo Preditivo,Echo,Concluído,09/12/2024,02/27/2025,2025-03-12,338,"61.970,49"


### **Ultimas Linhas do Dataset**

,projeto_id,cliente_id,nome_projeto,tipo_servico,squad,status,data_inicio,data_fim_prevista,data_fim_real,horas_vendidas,valor_contrato
136,PRJ0137,CLI042,Treinamento - Ébano,Treinamento,Alpha,Concluído,2025-06-20,14-jul-2025,2025-07-14,97,"13.899,29"
137,PRJ0138,CLI028,Treinamento - Rubi,Treinamento,Echo,Em andamento,08/05/2026,06/23/2026,NaN,156,25492.05
138,PRJ0139,CLI023,Modelo Preditivo - Tucano,Modelo Preditivo,Echo,Concluído,20-set-2024,2024-12-19,2024-12-16,325,NaN
139,PRJ0140,CLI033,Data Warehouse - Xingu,Data Warehouse,Foxtrot,Concluído,08/25/2025,2026-02-06,2026-02-09,572,118718.29
140,PRJ0010,CLI001,Modelo Preditivo - Umbu,Modelo Preditivo,Alpha,Concluído,2024-07-08,22/10/2024,10/21/2024,578,98986.73


### **Informações do Dataset**

...

,0
projeto_id,140
cliente_id,54
nome_projeto,87
tipo_servico,20
squad,6
status,5
data_inicio,132
data_fim_prevista,131
data_fim_real,117
horas_vendidas,124


### **Estatísticas Descritivas**

,horas_vendidas
count,141.000000
mean,327.382979
std,225.486447
min,51.000000
25%,149.000000
50%,285.000000
75%,430.000000
max,1030.000000


### **Os tipos de dados**

,0
projeto_id,object
cliente_id,object
nome_projeto,object
tipo_servico,object
squad,object
status,object
data_inicio,object
data_fim_prevista,object
data_fim_real,object
horas_vendidas,int64


In [12]:
# @title ## Informações Iniciais do dataset - df_apontamentos
display(Markdown('### **Primeiras Linhas do Dataset**'))
display(df_apontamentos.head())

display(Markdown('### **Ultimas Linhas do Dataset**'))
display(df_apontamentos.tail())

display(Markdown('### **Informações do Dataset**'))
display(df_apontamentos.info())

display(Markdown('### **Quantidade de Linhas e Colunas do Dataset**'))
display(df_apontamentos.shape)

display(Markdown('### **Quantidade de Valores Ausentes**'))
display(df_apontamentos.isnull().sum())

display(Markdown('### **Quantidade de Valores Duplicados**'))
display(df_apontamentos.duplicated().sum())

display(Markdown('### **Quantidade de Valores Únicos**'))
display(df_apontamentos.nunique())

display(Markdown('### **Estatísticas Descritivas**'))
display(df_apontamentos.describe())

display(Markdown('### **Os tipos de dados**'))
display(df_apontamentos.dtypes)

### **Primeiras Linhas do Dataset**

,apontamento_id,projeto_id,analista_id,data,horas,atividade
0,APT002473,PRJ0032,ANL006,2026-03-23,6,Reunião com cliente
1,APT004798,PRJ0064,ANL008,2026-05-08,6.5,Apresentação
2,APT005566,PRJ0079,ANL012,2025-06-02,6,Coleta
3,APT000099,PRJ0003,ANL009,2025-09-15,4.5,Documentação
4,APT009367,PRJ0134,ANL009,2026-01-22,4,Modelagem


### **Ultimas Linhas do Dataset**

,apontamento_id,projeto_id,analista_id,data,horas,atividade
9874,APT004768,PRJ0064,ANL020,2026-04-24,2,Documentação
9875,APT001287,PRJ0014,ANL014,06-nov-2024,2,Apresentação
9876,APT007173,PRJ0102,ANL019,28/01/2025,2,Documentação
9877,APT008447,PRJ0120,ANL019,12/05/2026,4.5,Documentação
9878,APT003940,PRJ0050,ANL008,03-nov-2025,3,Documentação


### **Informações do Dataset**

...

,0
apontamento_id,9799
projeto_id,140
analista_id,24
data,2037
horas,23
atividade,15


### **Estatísticas Descritivas**

,apontamento_id,projeto_id,analista_id,data,horas,atividade
count,9879,9879,9879,9879,9879,9879
unique,9799,140,24,2037,23,15
top,APT002135,PRJ0042,ANL005,2026-06-08,4,Coleta
freq,2,273,538,57,1667,1232


### **Os tipos de dados**

,0
apontamento_id,object
projeto_id,object
analista_id,object
data,object
horas,object
atividade,object


In [13]:
# @title ## Informações Iniciais do dataset - df_satisfacao
display(Markdown('### **Primeiras Linhas do Dataset**'))
display(df_satisfacao.head())

display(Markdown('### **Ultimas Linhas do Dataset**'))
display(df_satisfacao.tail())

display(Markdown('### **Informações do Dataset**'))
display(df_satisfacao.info())

display(Markdown('### **Quantidade de Linhas e Colunas do Dataset**'))
display(df_satisfacao.shape)

display(Markdown('### **Quantidade de Valores Ausentes**'))
display(df_satisfacao.isnull().sum())

display(Markdown('### **Quantidade de Valores Duplicados**'))
display(df_satisfacao.duplicated().sum())

display(Markdown('### **Quantidade de Valores Únicos**'))
display(df_satisfacao.nunique())

display(Markdown('### **Estatísticas Descritivas**'))
display(df_satisfacao.describe())

display(Markdown('### **Os tipos de dados**'))
display(df_satisfacao.dtypes)

### **Primeiras Linhas do Dataset**

,pesquisa_id,projeto_id,data_pesquisa,nota_nps,comentario
0,PSQ0001,PRJ0001,2024-05-27,NaN,"Resultado ok, prazo apertado."
1,PSQ0002,PRJ0003,2025-10-20,10.0,Documentação impecável.
2,PSQ0003,PRJ0004,2024-05-08,9.0,Documentação impecável.
3,PSQ0004,PRJ0005,2025-03-27,7.0,"Bom projeto, comunicação pode melhorar."
4,PSQ0005,PRJ0006,17/07/2025,6.0,Suporte demorou a responder.


### **Ultimas Linhas do Dataset**

,pesquisa_id,projeto_id,data_pesquisa,nota_nps,comentario
98,PSQ0099,PRJ0134,2026-02-26,NaN,"Resultado ok, prazo apertado."
99,PSQ0100,PRJ0135,2024-04-19,10.0,O painel virou rotina da diretoria.
100,PSQ0101,PRJ0137,2025-07-22,9.0,O painel virou rotina da diretoria.
101,PSQ0102,PRJ0139,2024-12-23,7.0,"Bom projeto, comunicação pode melhorar."
102,PSQ0103,PRJ0140,13/02/2026,8.0,"Bom projeto, comunicação pode melhorar."


### **Informações do Dataset**

...

,0
pesquisa_id,103
projeto_id,103
data_pesquisa,100
nota_nps,8
comentario,10


### **Estatísticas Descritivas**

,nota_nps
count,95.000000
mean,8.421053
std,1.601721
min,3.000000
25%,8.000000
50%,9.000000
75%,10.000000
max,10.000000


### **Os tipos de dados**

,0
pesquisa_id,object
projeto_id,object
data_pesquisa,object
nota_nps,float64
comentario,object
